# Distributed XGBoost training on Cover Type with Ray Train

This notebook trains a 7-class XGBoost model on the UCI Cover Type dataset (581,012 rows, 54 features) with Ray Train, scores it with Ray Data, and resumes training from a checkpoint.

Every Ray node mounts the cluster's storage directory at `STORAGE_ROOT`. The dataset goes to `STORAGE_ROOT/covtype/` and Ray Train's checkpoints go to `STORAGE_ROOT/checkpoints/`. The kernel itself never reads that directory; Ray tasks do. The notebook therefore runs unchanged with a kernel in a task on the cluster or with a kernel on another machine that connects through Ray Client.

The kernel finds the cluster through the environment variable `RAY_ADDRESS`. A Ray job on the head has it set. A kernel on another machine needs `RAY_ADDRESS=ray://<head address>:<Ray Client port>`, `RAY_AUTH_MODE=token` and `RAY_AUTH_TOKEN`, and the same Python and package versions as the cluster's image.

In [ ]:
# 00. Settings
import os

os.environ["RAY_TRAIN_V2_ENABLED"] = "1"

STORAGE_ROOT = "/mnt/cluster_storage/ray-demo"
NUM_WORKERS = 2        # Ray Train workers
CPUS_PER_WORKER = 2    # XGBoost threads per worker; the Ray node's other CPUs serve Ray Data
NUM_BOOST_ROUND = 50   # boosting rounds per fit

PARQUET_DIR = os.path.join(STORAGE_ROOT, "covtype")
TRAIN_PARQUET = os.path.join(PARQUET_DIR, "train.parquet")
VAL_PARQUET = os.path.join(PARQUET_DIR, "val.parquet")
CHECKPOINT_DIR = os.path.join(STORAGE_ROOT, "checkpoints")

In [ ]:
# 01. Imports and connection to the cluster
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix
import xgboost as xgb
import pyarrow as pa

import ray
import ray.data as rd
from ray.data import ActorPoolStrategy
from ray.train import RunConfig, ScalingConfig, CheckpointConfig, FailureConfig, get_dataset_shard, get_checkpoint, get_context
from ray.train.xgboost import XGBoostTrainer, RayTrainReportCallback

ray.init(ignore_reinit_error=True, logging_level="ERROR")   # the address comes from RAY_ADDRESS
print("Ray", ray.__version__, {k: v for k, v in ray.cluster_resources().items() if k in ("CPU", "memory")})

## Prepare the dataset

A Ray task downloads Cover Type once and writes a stratified 80/20 split as two Parquet files. Later runs find the files and skip the download.

In [ ]:
# 02. Download Cover Type and write train/validation Parquet files
@ray.remote(num_cpus=1)
def prepare_dataset(parquet_dir, train_parquet, val_parquet):
    import os
    from sklearn.datasets import fetch_covtype
    from sklearn.model_selection import train_test_split

    if os.path.exists(train_parquet) and os.path.exists(val_parquet):
        return "already present"
    df = fetch_covtype(as_frame=True).frame
    df.rename(columns={"Cover_Type": "label"}, inplace=True)   # Ray expects "label"
    df["label"] = df["label"] - 1                               # 1-7 -> 0-6
    assert df["label"].between(0, 6).all()
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
    os.makedirs(parquet_dir, exist_ok=True)
    train_df.to_parquet(train_parquet, index=False)
    val_df.to_parquet(val_parquet, index=False)
    return f"wrote {len(train_df):,} train rows and {len(val_df):,} validation rows"

print(ray.get(prepare_dataset.remote(PARQUET_DIR, TRAIN_PARQUET, VAL_PARQUET)))

In [ ]:
# 03. Load the two splits as Ray Datasets (lazy, columnar)
train_ds = rd.read_parquet(TRAIN_PARQUET).random_shuffle()
val_ds = rd.read_parquet(VAL_PARQUET)
print(f"Train rows: {train_ds.count():,},  Val rows: {val_ds.count():,}")

In [ ]:
# 04. Class distribution of the validation split
val_pd = val_ds.to_pandas()
val_pd.label.value_counts().sort_index().plot(kind="bar", figsize=(6, 3), title="Cover Type distribution (validation split)")
plt.ylabel("Frequency"); plt.show()
feature_columns = [c for c in val_pd.columns if c != "label"]
print(len(feature_columns), "features")

## Train

Each Ray Train worker collects its shard of the dataset into an XGBoost `DMatrix`, resumes from a checkpoint when Ray Train supplies one, and reports metrics every boosting round. A checkpoint is written every 10 rounds.

In [ ]:
# 05. Training function run by every Ray Train worker
INDEX_COLS = {"__index_level_0__"}

def _arrow_table_from_shard(name: str) -> pa.Table:
    tables = [ray.get(r) for r in get_dataset_shard(name).materialize().to_arrow_refs()]
    tbl = pa.concat_tables(tables, promote_options="none") if tables else pa.table({})
    return tbl.select([c for c in tbl.column_names if c not in INDEX_COLS])

def _dmat_from_arrow(table: pa.Table, feature_cols, label_col: str):
    X = np.column_stack([table[c].to_numpy(zero_copy_only=False) for c in feature_cols])
    y = table[label_col].to_numpy(zero_copy_only=False)
    return xgb.DMatrix(X, label=y, feature_names=feature_cols)

def train_func(config):
    label_col = config["label_column"]
    train_arrow = _arrow_table_from_shard("train")
    eval_arrow = _arrow_table_from_shard("evaluation")
    feature_cols = [c for c in train_arrow.column_names if c != label_col]
    dtrain = _dmat_from_arrow(train_arrow, feature_cols, label_col)
    deval = _dmat_from_arrow(eval_arrow, feature_cols, label_col)

    booster = None
    ckpt = get_checkpoint()
    if ckpt:
        with ckpt.as_directory() as d:
            booster = xgb.Booster()
            booster.load_model(os.path.join(d, RayTrainReportCallback.CHECKPOINT_NAME))
            print(f"[Rank {get_context().get_world_rank()}] Resumed from checkpoint")

    xgb.train(
        params=config["params"],
        dtrain=dtrain,
        evals=[(dtrain, "train"), (deval, "validation")],
        num_boost_round=config["num_boost_round"],
        xgb_model=booster,
        callbacks=[RayTrainReportCallback(frequency=10)],   # checkpoint every 10 rounds
    )

In [ ]:
# 06. XGBoost parameters and the trainer
xgb_params = {
    "objective": "multi:softprob",
    "num_class": 7,
    "eval_metric": "mlogloss",
    "tree_method": "hist",
    "eta": 0.3,
    "max_depth": 8,
    "nthread": CPUS_PER_WORKER,
}

trainer = XGBoostTrainer(
    train_func,
    scaling_config=ScalingConfig(num_workers=NUM_WORKERS, use_gpu=False, resources_per_worker={"CPU": CPUS_PER_WORKER}),
    datasets={"train": train_ds, "evaluation": val_ds},
    train_loop_config={"label_column": "label", "params": xgb_params, "num_boost_round": NUM_BOOST_ROUND},
    run_config=RunConfig(
        name="covtype_xgb_cpu",
        storage_path=CHECKPOINT_DIR,
        checkpoint_config=CheckpointConfig(num_to_keep=1, checkpoint_score_attribute="validation-mlogloss", checkpoint_score_order="min"),
        failure_config=FailureConfig(max_failures=1),
    ),
)

In [ ]:
# 07. Fit: NUM_BOOST_ROUND boosting rounds
result = trainer.fit()
best_ckpt = result.checkpoint
print(result.metrics.get("validation-mlogloss"), best_ckpt.path)

## Evaluate

In [ ]:
# 08. Fetch the checkpoint's Booster through a Ray task and score the validation split in the kernel
@ray.remote(num_cpus=1)
def load_model_bytes(ckpt):
    return bytes(RayTrainReportCallback.get_model(ckpt).save_raw("ubj"))

def load_booster(ckpt):
    b = xgb.Booster()
    b.load_model(bytearray(ray.get(load_model_bytes.remote(ckpt))))
    return b

booster = load_booster(best_ckpt)
pred_labels = np.argmax(booster.predict(xgb.DMatrix(val_pd[feature_columns])), axis=1)
print(f"Validation accuracy: {accuracy_score(val_pd.label, pred_labels):.3f}")

In [ ]:
# 09. Confusion matrix
cm = confusion_matrix(val_pd.label, pred_labels)
sns.heatmap(cm, annot=True, fmt="d", cmap="viridis")
plt.title("Confusion Matrix with Counts"); plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()

cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="viridis")
plt.title("Normalized Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()

In [ ]:
# 10. Batch inference with Ray Data: a pool of actors, each loading the Booster once
class XGBPredictor:
    def __init__(self, ckpt, feature_cols):
        self.model = RayTrainReportCallback.get_model(ckpt)
        self.feature_cols = feature_cols

    def __call__(self, batch: pd.DataFrame) -> pd.DataFrame:
        preds = np.argmax(self.model.predict(xgb.DMatrix(batch[self.feature_cols])), axis=1)
        return pd.DataFrame({"pred": preds.astype(np.int32), "label": batch["label"].astype(np.int32)})

def ray_data_accuracy(ckpt):
    pred_ds = val_ds.map_batches(XGBPredictor, fn_constructor_args=(ckpt, feature_columns), batch_format="pandas",
                                 compute=ActorPoolStrategy(), num_cpus=1)
    stats_ds = pred_ds.map_batches(
        lambda df: pd.DataFrame({"correct": [int((df["pred"].to_numpy() == df["label"].to_numpy()).sum())], "n": [int(len(df))]}),
        batch_format="pandas")
    return int(stats_ds.sum("correct")) / int(stats_ds.sum("n"))

print(f"Validation accuracy (Ray Data inference): {ray_data_accuracy(best_ckpt):.3f}")

In [ ]:
# 11. Gain-based feature importance
importances = booster.get_score(importance_type="gain")
keys, gains = zip(*sorted(importances.items(), key=lambda kv: kv[1], reverse=True)[:15])
plt.barh(range(len(gains)), gains); plt.yticks(range(len(gains)), keys); plt.gca().invert_yaxis()
plt.title("Top-15 Feature Importances (gain)"); plt.xlabel("Average gain"); plt.show()

## Continue training from the checkpoint

A second `fit()` under the same run name starts from the last checkpoint and adds `NUM_BOOST_ROUND` rounds.

In [ ]:
# 12. Run NUM_BOOST_ROUND more boosting rounds from the last checkpoint
result = trainer.fit()
best_ckpt = result.checkpoint
print(result.metrics.get("validation-mlogloss"), best_ckpt.path)

In [ ]:
# 13. Accuracy after the continued training
print(f"Validation accuracy after continued training: {ray_data_accuracy(best_ckpt):.3f}")

In [ ]:
# 14. Cleanup: delete this run's checkpoints, so the next run of cell 07 starts from round 0.
# The Parquet files stay; delete PARQUET_DIR as well to repeat the download.
@ray.remote(num_cpus=1)
def delete_run(path):
    import shutil
    shutil.rmtree(path, ignore_errors=True)
    return path

print("Deleted", ray.get(delete_run.remote(os.path.join(CHECKPOINT_DIR, "covtype_xgb_cpu"))))